In [ ]:
import pandas as pd
import numpy as np
import equiboots as eqb
from equiboots.tables import metrics_table
from core.model_registry import best_per_algo, load_best_per_algo

In [ ]:
help(eqb)

## Read in Data and Model Object

In [ ]:
X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")
model_catboost = champs["cat_outcome"]

In [ ]:
X_valid, y_valid = model_catboost.get_valid_data(X, y)
X_test, y_test = model_catboost.get_test_data(X, y)
y_test = y_test["outcome"]

## Point Estimates

In [ ]:
thr = model_catboost.threshold["average_precision"]
# get predictions and true values
y_prob = model_catboost.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= thr).astype(int)
y_test = y_test.to_numpy()


In [ ]:
X_test = X_test.copy()
X_test["sex"] = X_test["sex"].map({1: "Male", 0: "Female"})
X_test[["sex"]] = X_test[["sex"]].astype(str)


# Create fairness DataFrame
fairness_df = X_test[["sex"]].reset_index()

eq = eqb.EquiBoots(
    y_true=y_test,
    y_prob=y_prob,
    y_pred=y_pred,
    fairness_df=fairness_df,
    fairness_vars=["sex"],
)

# grouping by variables' groups (e.g., Male, Female, etc)
eq.grouper(groupings_vars=["sex"])

In [ ]:

# slicing and generating metrics for sex
sliced_sex_data = eq.slicer("sex")
sex_metrics = eq.get_metrics(sliced_sex_data)

In [ ]:
## generating statistical significnace in tests
test_config = {
    "test_type": "chi_square",
    "alpha": 0.05,
    "adjust_method": "bonferroni",
    "confidence_level": 0.95,
    "classification_task": "binary_classification",
}

# stat test sex
stat_test_results_sex = eq.analyze_statistical_significance(
    sex_metrics, "sex", test_config,
)

In [ ]:
stat_test_results_sex

In [ ]:
overall_stat_results = {
    "sex": stat_test_results_sex,
}

In [ ]:
eqb.eq_group_metrics_point_plot(
    group_metrics=[sex_metrics],
    metric_cols=[
        "ROC AUC",
        "Precision",
        "Recall",
    ],
    category_names=["sex"],
    figsize=(6, 8),
    include_legend=True,
    plot_thresholds=(0.9, 1.1),
    raw_metrics=True,
    show_grid=True,
    y_lim=(0, 1),
    statistical_tests=overall_stat_results,
    y_lims={(0, 0): (0.70, 1.0), (0, 1): (0.70, 1.0)},
)

In [ ]:
stat_metrics_table_point = metrics_table(
    sex_metrics, statistical_tests=stat_test_results_sex, reference_group="Male",
)

In [ ]:
stat_metrics_table_point

In [ ]:
eqb.eq_plot_metrics_forest(
    group_metrics=sex_metrics,
    metric_name="ROC AUC",
    title="Forest Plot: ROC AUC Across Groups",
    reference_group="Male",
    statistical_tests=stat_test_results_sex,
)

In [ ]:
eqb.plot_effect_sizes(
    stat_test_results_sex,
    xlabel="Sex",
    ylabel="Effect size",
    title="Sex Effect Sizes",
    figsize=(10, 4),
)

## Precision-Recall, ROC, and Calibration Curves by Sex
These plots look at how performance is different across the different sex groups.
We choose to exclude certain groups from the analysis because there are not enough members of these groups to make a
fair comparison between the groups.

In [ ]:
# PR curves
eqb.eq_plot_group_curves(
    sliced_sex_data,
    curve_type="pr",
    subplots=False,
    figsize=(7, 7),
    title="Precision-Recall by Sex Group",
)

In [ ]:
# PR curves
eqb.eq_plot_group_curves(
    sliced_sex_data,
    curve_type="roc",
    subplots=False,
    figsize=(7, 7),
    title="ROC AUC by Sex Group",
)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)

# Loop variables are deliberately named yt/yp rather than y_true/y_prob so
# they do not shadow the module-level arrays used elsewhere in the notebook.

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5.5))

colors = {"Female": "#1f77b4", "Male": "#ff7f0e"}

for group, d in sliced_sex_data.items():
    yt, yp = d["y_true"], d["y_prob"]
    n, pos = len(yt), int(yt.sum())
    c = colors.get(group)

    fpr, tpr, _ = roc_curve(yt, yp)
    ax_roc.plot(
        fpr,
        tpr,
        color=c,
        label=f"{group}: AUC = {auc(fpr, tpr):.2f} (n = {n}, {pos} events)",
    )

    prec, rec, _ = precision_recall_curve(yt, yp)
    ax_pr.plot(
        rec,
        prec,
        color=c,
        label=f"{group}: AP = {average_precision_score(yt, yp):.2f} "
              f"(n = {n}, {pos} events)",
    )

ax_roc.plot([0, 1], [0, 1], "k--", linewidth=1)
ax_roc.set_xlabel("False positive rate")
ax_roc.set_ylabel("True positive rate")
ax_roc.set_title("(a) ROC by sex", loc="left", fontweight="bold")

ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("(b) Precision-recall by sex", loc="left", fontweight="bold")

for ax in (ax_roc, ax_pr):
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

plt.tight_layout()

for path in ("../images/png_images/figure2_group_curves.png",
             "../images/pdf_images/figure2_group_curves.pdf"):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

fig.savefig("../images/png_images/figure2_group_curves.png",
            dpi=300, bbox_inches="tight")
fig.savefig("../images/pdf_images/figure2_group_curves.pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
# calibration curves
eqb.eq_plot_group_curves(
    sliced_sex_data,
    curve_type="calibration",
    shade_area=True,
    title="Calibration by Sex Group",
    subplots=False,
)

## Bootstrap Estimates

Bootstrap estimates:m
- randomly sampling fairness_df, y_true, y_prob, and y_pred

In [ ]:
# setting fixed seed for reproducibility
# Alternatively, seeds can be set after initialization
int_list = np.linspace(0, len(y_test), num=len(y_test), dtype=int).tolist()

eq2 = eqb.EquiBoots(
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    fairness_df=fairness_df,
    fairness_vars=["sex"],
    seeds=int_list,
    reference_groups=["Male"],
    task="binary_classification",
    bootstrap_flag=True,
    num_bootstraps=5001,
    boot_sample_size=len(y_test),  # whole length of test set
    group_min_size=1,  # any group with samples below this number will be ignored
    balanced=False,  # False is stratified (i.e., maintaining groups proportions), True is balanced (equal proportions)
    stratify_by_outcome=True,  # True maintain initial dataset outcome proportions per group
)

# Set seeds after initialization
eq2.set_fix_seeds(int_list)
print("seeds", eq2.seeds)

# group bootstraps by grouping variables (e.g., sex)
eq2.grouper(groupings_vars=["sex"])

# slice by variable and assign to a variable
# sex related bootstraps
boots_sex_data = eq2.slicer("sex")

### Calculate disparities

In [ ]:
# compute binary classification metrics wrt to sex
boots_sex_metrics = eq2.get_metrics(boots_sex_data)

In [ ]:
dispa = eq2.calculate_disparities(boots_sex_metrics, "sex")

In [ ]:
eqb.eq_group_metrics_plot(
    group_metrics=dispa,
    metric_cols=[
        "Accuracy_Ratio",
        "Precision_Ratio",
        "Predicted_Prevalence_Ratio",
        "Prevalence_Ratio",
        "FP_Rate_Ratio",
        "TN_Rate_Ratio",
        "Recall_Ratio",
    ],
    name="sex",
    categories="all",
    plot_type="violinplot",
    color_by_group=True,
    show_grid=False,
    strict_layout=True,
    leg_cols=7,
    # plot_thresholds=[0.9, 1.2],
)

In [ ]:
diffs = eq2.calculate_differences(boots_sex_metrics, "sex")

In [ ]:
# metrics to perform a statistical test
metrics_boot = [
    "Accuracy_diff",
    "Precision_diff",
    "Recall_diff",
    "F1_Score_diff",
    "Specificity_diff",
    "TP_Rate_diff",
    "FP_Rate_diff",
    "FN_Rate_diff",
    "TN_Rate_diff",
    "Prevalence_diff",
    "Predicted_Prevalence_diff",
    "ROC_AUC_diff",
    "Average_Precision_Score_diff",
    "Log_Loss_diff",
    "Brier_Score_diff",
    "Calibration_AUC_diff",
]

# configuration dictionary to provide parameters around statistical testing
test_config = {
    "test_type": "bootstrap_test",
    "alpha": 0.05,
    "adjust_method": "bonferroni",
    "confidence_level": 0.95,
    "classification_task": "binary_classification",
    "tail_type": "two_tailed",
    "metrics": metrics_boot,
}


stat_test_results = eq2.analyze_statistical_significance(
    metric_dict=boots_sex_metrics,  # pass variable sliced metrics
    var_name="sex",  # variable name
    test_config=test_config,  # configuration
    differences=diffs,  # the differences of each sex group
)

In [ ]:
stat_metrics_table_diff = metrics_table(
    boots_sex_metrics,
    statistical_tests=stat_test_results,
    differences=diffs,
    reference_group="Male",
)

In [ ]:
# differences of each sex group wrt reference group
# reference group differences are all zero not shown for simplicity
# * depicts statistical significance
stat_metrics_table_diff

In [ ]:
wanted = {"ROC_AUC_diff", "FP_Rate_diff", "Predicted_Prevalence_diff", "Recall_diff"}
wanted_metrics = [x for x in metrics_boot if x in wanted]

In [ ]:
eqb.eq_group_metrics_plot(
    group_metrics=diffs,
    metric_cols=wanted_metrics,
    name="sex",
    categories="all",
    figsize=(8,8),
    plot_type="violinplot",
    color_by_group=True,
    max_cols=2,
    show_grid=False,
    strict_layout=True,
    reference_group="Male",
    save_path="../images",
    show_pass_fail=False,
    statistical_tests=stat_test_results,
)

In [ ]:
eqb.eq_plot_bootstrap_forest(
    group_boot_metrics=boots_sex_metrics,
    metric="ROC AUC",
    reference_group="Male",
    title="AUROC - Bootstrapped Sex Metrics",
    figsize=(8, 6),
    statistical_tests= stat_test_results
    )

In [ ]:
eqb.calculate_bootstrap_stats(group_boot_metrics=boots_sex_metrics, metric="ROC AUC")

In [ ]:
eqb.eq_plot_bootstrapped_group_curves(
    boot_sliced_data=boots_sex_data,
    curve_type="roc",
    # title="Bootstrapped ROC Curve by Sex",
    title="",
    bar_every=100,
    subplots=True,
    n_bins=10,
    figsize=(5, 5),
    color_by_group=True,
    save_path = "../images/pdf_images",
    filename="bootstrapped_roc_curve_by_sex.pdf"
)